In [35]:
import pandas as pd
from scipy.linalg import svd

In [2]:
ratings = pd.read_csv("rating.csv")
movies = pd.read_csv('movie.csv')

In [3]:
df = pd.merge(ratings, movies, how='left', on='movieId')
df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),Adventure|Children|Fantasy
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [4]:
df[df['movieId'].isin(df['movieId'].value_counts().index[:5])]['title'].unique()

array(['Pulp Fiction (1994)', 'Shawshank Redemption, The (1994)',
       'Silence of the Lambs, The (1991)', 'Jurassic Park (1993)',
       'Forrest Gump (1994)'], dtype=object)

In [5]:
df.groupby('title')['rating'].mean().sort_values(ascending=False)

title
Small Roads (2011)                   5.0
Divorce (1945)                       5.0
The Beautiful Story (1992)           5.0
Into the Middle of Nowhere (2010)    5.0
The Sea That Thinks (2000)           5.0
                                    ... 
I Dream Too Much (1935)              0.5
Cameron's Closet (1988)              0.5
Double Trouble (1992)                0.5
Through the Forest (2005)            0.5
Slumber Party '57 (1976)             0.5
Name: rating, Length: 26729, dtype: float64

In [6]:
popular_movies = df['movieId'].value_counts()
popular_movies = popular_movies[popular_movies > 50].index.tolist()

In [7]:
filtered_df = df[df['movieId'].isin(popular_movies)]
filtered_df.groupby('title')['rating'].mean().sort_values(ascending=False)

title
Shawshank Redemption, The (1994)       4.446990
Godfather, The (1972)                  4.364732
Usual Suspects, The (1995)             4.334372
Schindler's List (1993)                4.310175
Godfather: Part II, The (1974)         4.275641
                                         ...   
Barney's Great Adventure (1998)        1.163484
Glitter (2001)                         1.124088
Bratz: The Movie (2007)                1.105556
From Justin to Kelly (2003)            0.973005
SuperBabies: Baby Geniuses 2 (2004)    0.837321
Name: rating, Length: 10472, dtype: float64

In [8]:
articles = pd.read_csv('shared_articles.csv')
articles.head()

,timestamp,eventType,contentId,authorPersonId,authorSessionId,authorUserAgent,authorRegion,authorCountry,contentType,url,title,text,lang
0,1459192779,CONTENT REMOVED,-6451309518266745024,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://www.nytimes.com/2016/03/28/business/dea...,"Ethereum, a Virtual Currency, Enables Transact...",All of this work is still very early. The firs...,en
1,1459193988,CONTENT SHARED,-4110354420726924665,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://www.nytimes.com/2016/03/28/business/dea...,"Ethereum, a Virtual Currency, Enables Transact...",All of this work is still very early. The firs...,en
2,1459194146,CONTENT SHARED,-7292285110016212249,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,http://cointelegraph.com/news/bitcoin-future-w...,Bitcoin Future: When GBPcoin of Branson Wins O...,The alarm clock wakes me at 8:00 with stream o...,en
3,1459194474,CONTENT SHARED,-6151852268067518688,3891637997717104548,-1457532940883382585,NaN,NaN,NaN,HTML,https://cloudplatform.googleblog.com/2016/03/G...,Google Data Center 360° Tour,We're excited to share the Google Data Center ...,en
4,1459194497,CONTENT SHARED,2448026894306402386,4340306774493623681,8940341205206233829,NaN,NaN,NaN,HTML,https://bitcoinmagazine.com/articles/ibm-wants...,"IBM Wants to ""Evolve the Internet"" With Blockc...",The Aite Group projects the blockchain market ...,en


In [9]:
articles_df = articles[articles['eventType'] == 'CONTENT SHARED']
articles_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3047 entries, 1 to 3121
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   timestamp        3047 non-null   int64 
 1   eventType        3047 non-null   object
 2   contentId        3047 non-null   int64 
 3   authorPersonId   3047 non-null   int64 
 4   authorSessionId  3047 non-null   int64 
 5   authorUserAgent  669 non-null    object
 6   authorRegion     669 non-null    object
 7   authorCountry    669 non-null    object
 8   contentType      3047 non-null   object
 9   url              3047 non-null   object
 10  title            3047 non-null   object
 11  text             3047 non-null   object
 12  lang             3047 non-null   object
dtypes: int64(4), object(9)
memory usage: 333.3+ KB


In [10]:
interactions_df = pd.read_csv('users_interactions.csv')
interactions_df.personId = interactions_df.personId.astype(str)
interactions_df.contentId = interactions_df.contentId.astype(str)
articles_df.contentId = articles_df.contentId.astype(str)

/var/folders/6k/1096gq_54_dgk0_cjz4762g40000gn/T/ipykernel_96340/2391869325.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  articles_df.contentId = articles_df.contentId.astype(str)


In [11]:
event_type = {
    'VIEW': 1.0,
    'LIKE': 2.0,
    'BOOKMARK': 2.5,
    'FOLLOW': 3.0,
    'COMMENT CREATED': 4.0,
}

In [12]:
interactions_df.head()

,timestamp,eventType,contentId,personId,sessionId,userAgent,userRegion,userCountry
0,1465413032,VIEW,-3499919498720038879,-8845298781299428018,1264196770339959068,NaN,NaN,NaN
1,1465412560,VIEW,8890720798209849691,-1032019229384696495,3621737643587579081,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,NY,US
2,1465416190,VIEW,310515487419366995,-1130272294246983140,2631864456530402479,NaN,NaN,NaN
3,1465413895,FOLLOW,310515487419366995,344280948527967603,-3167637573980064150,NaN,NaN,NaN
4,1465412290,VIEW,-7820640624231356730,-445337111692715325,5611481178424124714,NaN,NaN,NaN


In [13]:
interactions_df['eventWeight'] = interactions_df['eventType'].map(event_type)

In [14]:
interactions_df['eventWeight'].mean()

1.2362885828078327

In [15]:
person_interactions = interactions_df.groupby('personId')['contentId'].nunique()
person_interactions = person_interactions[person_interactions >= 5].index.tolist()

In [16]:
interactions_df = interactions_df[interactions_df['personId'].isin(person_interactions)]
interactions_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69868 entries, 0 to 72311
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   timestamp    69868 non-null  int64  
 1   eventType    69868 non-null  object 
 2   contentId    69868 non-null  object 
 3   personId     69868 non-null  object 
 4   sessionId    69868 non-null  int64  
 5   userAgent    54877 non-null  object 
 6   userRegion   54867 non-null  object 
 7   userCountry  54877 non-null  object 
 8   eventWeight  69868 non-null  float64
dtypes: float64(1), int64(2), object(6)
memory usage: 5.3+ MB


In [17]:
import math


def smooth_user_preference(x):
    return math.log(1 + x, 2)

In [18]:
user_interactions = (
    interactions_df.groupby(['personId', 'contentId'], as_index=False)
    .agg(
        eventWeight=('eventWeight', 'sum'),
        lastInteraction=('timestamp', 'max'))
)
user_interactions['eventWeight'] = user_interactions['eventWeight'].apply(smooth_user_preference)
user_interactions

,personId,contentId,eventWeight,lastInteraction
0,-1007001694607905623,-5065077552540450930,1.000000,1470395911
1,-1007001694607905623,-6623581327558800021,1.000000,1487240080
2,-1007001694607905623,-793729620925729327,1.000000,1472834892
3,-1007001694607905623,1469580151036142903,1.000000,1487240062
4,-1007001694607905623,7270966256391553686,1.584963,1485994342
...,...,...,...,...
39101,998688566268269815,-401664538366009049,1.000000,1474567449
39102,998688566268269815,3456674717452933449,2.584963,1478802088
39103,998688566268269815,6881796783400625893,1.000000,1474567675
39104,998688566268269815,7174452660053929140,2.321928,1478812905


In [19]:
user_interactions['lastInteraction'].mean()

1470605340.0403006

In [20]:
interactions_train_df = user_interactions[user_interactions['lastInteraction'] < 1475519545]
interactions_test_df = user_interactions[user_interactions['lastInteraction'] >= 1475519545]
interactions_train_df.shape, interactions_test_df.shape

((29325, 4), (9781, 4))

In [21]:
import numpy as np

final_df = (
    interactions_train_df.reset_index()
    .groupby('personId')['contentId'].agg(lambda x: list(x))
    .reset_index()
    .rename(columns={'contentId': 'true_train'})
    .set_index('personId')
)

final_df['true_test'] = (
    interactions_test_df.reset_index()
    .groupby('personId')['contentId'].agg(lambda x: list(x))
)

final_df['true_test'] = [ [] if x is np.nan else x for x in final_df['true_test'] ]
final_df.head()

,true_train,true_test
personId,,
-1007001694607905623,"[-5065077552540450930, -793729620925729327]","[-6623581327558800021, 1469580151036142903, 72..."
-1032019229384696495,"[-1006791494035379303, -1039912738963181810, -...","[-1415040208471067980, -2555801390963402198, -..."
-108842214936804958,"[-1196068832249300490, -133139342397538859, -1...","[-2780168264183400543, -3060116862184714437, -..."
-1130272294246983140,"[-1150591229250318592, -1196068832249300490, -...","[-1606980109000976010, -1663441888197894674, -..."
-1160159014793528221,"[-133139342397538859, -387651900461462767, 377...",[-3462051751080362224]


In [22]:
popular = (
    interactions_train_df
    .groupby('contentId')
    .eventWeight.sum().reset_index()
    .sort_values('eventWeight', ascending=False)
    .contentId.values
)
popular[0]

'-6783772548752091658'

In [23]:
top_k = 10

final_df['popular'] = (
    final_df.true_train
    .apply(
        lambda x:
        popular[~np.isin(popular, x)][:top_k]
    )
)
def calc_precision(column):
    return (
        final_df
        .apply(
            lambda row:
            len(set(row['true_test']).intersection(
                set(row[column]))) /
            min(len(row['true_test']) + 0.001, 10.0),
            axis=1)).mean()
calc_precision('popular')

0.006454207722621089

In [24]:
ratings = pd.pivot_table(
    interactions_train_df,
    values="eventWeight",
    index="personId",
    columns="contentId",
).fillna(0)
round(ratings.loc["-1032019229384696495", "943818026930898372"], 2)

2.32

In [25]:
ratings_m = ratings.values
ratings_m.mean()

0.016668620737604063

In [26]:
similarity_users = np.zeros((len(ratings_m), len(ratings_m)))
for i in (range(len(ratings_m)-1)):
    for j in range(i+1, len(ratings_m)):

        mask_uv = (ratings_m[i] != 0) & (ratings_m[j] != 0)
        ratings_v = ratings_m[i, mask_uv]
        ratings_u = ratings_m[j, mask_uv]

        similarity_users[i,j] = np.corrcoef(ratings_v, ratings_u)[0, 1]
        similarity_users[j,i] = similarity_users[i,j]

similarity_users[3,40]

/Users/dmitryvokhmin/Desktop/courses/DS/ds-python-8/.venv/lib/python3.11/site-packages/numpy/lib/function_base.py:518: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/Users/dmitryvokhmin/Desktop/courses/DS/ds-python-8/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/dmitryvokhmin/Desktop/courses/DS/ds-python-8/.venv/lib/python3.11/site-packages/numpy/lib/function_base.py:2846: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/dmitryvokhmin/Desktop/courses/DS/ds-python-8/.venv/lib/python3.11/site-packages/numpy/lib/function_base.py:2705: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/dmitryvokhmin/Desktop/courses/DS/ds-python-8/.venv/lib/python3.11/site-packages/numpy/lib/function_base.py:2705: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/

-0.3333333333333333

In [27]:
interactions = (
    interactions_train_df
    .groupby('personId')['contentId'].agg(lambda x: list(x))
    .reset_index()
    .rename(columns={'contentId': 'true_train'})
    .set_index('personId')
)

interactions['true_test'] = (
    interactions_test_df
    .groupby('personId')['contentId'].agg(lambda x: list(x))
)

interactions['true_test'] = [ [] if x is np.NaN else x for x in interactions['true_test'] ]

prediction_user_based = []
for i in range(len(similarity_users)):
    users_sim = similarity_users[i] > 0
    if not any(users_sim):
        prediction_user_based.append([])
    else:
        tmp_recommend = np.argsort(ratings_m[users_sim].sum(axis=0))[::-1]
        tmp_recommend = ratings.columns[tmp_recommend]
        recommend = np.array(tmp_recommend)[~np.in1d(tmp_recommend, interactions.iloc[i]["true_train"])][:10]
        prediction_user_based.append(list(recommend))
interactions['prediction_user_based'] = prediction_user_based
prediction_user_based[35][0]

'-5148591903395022444'

In [29]:
def calc_precision(column):
    return (
        interactions
        .apply(
            lambda row:
            len(set(row['true_test']).intersection(
                set(row[column]))) /
            min(len(row['true_test']) + 0.001, 10.0),
            axis=1)).mean()

In [30]:
round(calc_precision('prediction_user_based'), 3)

0.005

In [36]:
U, sigma, V = svd(ratings)
U.max()

0.7071067811865485

In [37]:
k = 100
s = np.diag(sigma[:k])
U = U[:, 0:k]
V = V[0:k, :]

round(s.sum(), 2)

2096.43

In [38]:
new_ratings = pd.DataFrame(
    U.dot(s).dot(V), index=ratings.index, columns=ratings.columns
)
top_k = 10
predictions = []

for personId in interactions.index:
    prediction = (
        new_ratings.loc[personId].sort_values(ascending=False).index.values
    )

    predictions.append(
        list(
            prediction[
                ~np.in1d(prediction, interactions.loc[personId, "true_train"])
            ]
        )[:top_k]
    )

interactions["prediction_svd"] = predictions

calc_precision("prediction_svd")

0.012023152289763838

In [39]:
from lightfm import LightFM
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k
from scipy.sparse import csr_matrix

ratings_matrix = csr_matrix(ratings) # передаём в качестве аргументов в функцию выставленный рейтинг (это будут значения матрицы), а также id пользователя и id книги (это будут индексы для строк и столбцов матрицы)
model = LightFM(loss='warp', #определяем функцию потерь
                random_state=13, #фиксируем случайное разбиение
                learning_rate=0.05, #темп обучения
                no_components=100) #размерность вектора для представления данных в модели

train,test = random_train_test_split(ratings_matrix, test_percentage=0.3, random_state=13)
model.fit(train)
prec_score = precision_at_k(model, test).mean()


print(round(prec_score, 2))

0.04


/Users/dmitryvokhmin/Desktop/courses/DS/ds-python-8/.venv/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(
